# RAG: Baseline Retrieval vs. Cross-Encoder Re-Ranking

**Document:** NIST SP 800-63B-4, *Digital Identity Guidelines: Authentication and Authenticator Management* (July 2025, 129 pages).
It is a regulatory text full of "SHALL", "SHALL NOT" and "except when" style rules, which makes it a good stress test for retrieval.

Both pipelines run over the **same chunks**:

| | Pipeline 1 (Baseline) | Pipeline 2 (Re-Ranking) |
|---|---|---|
| Candidate search | bi-encoder `all-MiniLM-L6-v2` + FAISS, top 10 by cosine | same top 10 |
| Ordering | cosine score | cross-encoder `ms-marco-MiniLM-L-6-v2` score |
| Sent to LLM | top 3 | top 3 after re-ranking |
| Prompt | `build_prompt()` | the same `build_prompt()` |

The LLM is Gemini (`gemini-3.8-flash`, temperature 0). The API key is read from the `GEMINI_API_KEY` environment variable.

In [1]:
# Transformers tries to import TensorFlow if it is installed; we only need PyTorch.
import os
os.environ["USE_TF"] = "0"

import bisect, collections, statistics, textwrap, time
import faiss
import numpy as np
import pandas as pd
import pymupdf
import torch
from google import genai
from google.genai import types
from sentence_transformers import CrossEncoder, SentenceTransformer

PDF_PATH = "NIST.SP.800-63B-4.pdf"
BI_ENCODER = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENCODER = "cross-encoder/ms-marco-MiniLM-L-6-v2"
LLM_MODEL = "gemini-3.8-flash"

TOP_K = 10            # candidates pulled from the vector index
TOP_N = 3             # chunks that actually go into the prompt
CHUNK_TOKENS = 200    # chunk size, measured in bi-encoder tokens
OVERLAP_TOKENS = 40   # shared tokens between neighbouring chunks

pd.set_option("display.max_colwidth", 90)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

Device: cpu


## Step 1: Load and clean the PDF

* The first 10 pages are the cover, abstract and table of contents. The TOC lines (e.g. *"Rate Limiting (Throttling) .... 41"*) look like real content to a retriever, so I start at the **Preface**.
* Every page repeats the same header (*"NIST SP 800-63B-4 / July 2025 / Digital Identity Guidelines ..."*). Any line that appears on more than half of the pages is treated as a running header and dropped.
* All pages are joined into one string, and I remember where each page starts so every chunk can be cited by page number.

In [2]:
pages = [page.get_text() for page in pymupdf.open(PDF_PATH)]
first_page = next(i for i, text in enumerate(pages) if "\nPreface\n" in text)

# Lines that repeat on most pages are headers/footers, not content.
line_counts = collections.Counter(line.strip() for text in pages for line in set(text.splitlines()))
boilerplate = {line for line, n in line_counts.items() if n > len(pages) / 2}
print("Removed header lines:", boilerplate)

full_text, page_starts = "", []
for i in range(first_page, len(pages)):
    page_starts.append(len(full_text))
    lines = [l.strip() for l in pages[i].splitlines() if l.strip() and l.strip() not in boilerplate]
    # Bullet glyphs come out as odd characters; collapse everything to single spaces.
    text = " ".join(lines).replace("\ufffd", " ").replace("\u2022", " ")
    full_text += " ".join(text.split()) + " "

print(f"Kept pages {first_page + 1}-{len(pages)}, {len(full_text):,} characters")

Removed header lines: {'July 2025', 'NIST SP 800-63B-4', 'Digital Identity Guidelines', 'Authentication and Authenticator Management'}
Kept pages 11-129, 257,138 characters


## Step 2: Chunking

The bi-encoder silently **truncates anything beyond 256 word-pieces**. NIST text is token-heavy (section numbers, "SP 800-63A", URLs), so my first try with 180-*word* windows produced chunks of up to 520 tokens, and the end of those chunks would have been invisible to the bi-encoder (but not to the cross-encoder, which reads 512). That would have made the comparison unfair.

So I chunk directly on the bi-encoder's tokens: windows of 200 tokens with 40 tokens of overlap. The tokenizer's offset mapping lets me slice the *original* text, so chunks keep their casing and punctuation.

In [3]:
bi_encoder = SentenceTransformer(BI_ENCODER)
tok = bi_encoder.tokenizer

# Tokenise the whole document once. verbose=False hides the "longer than 256" warning,
# which does not apply here because the full text is never fed to the model.
offsets = tok(full_text, add_special_tokens=False, return_offsets_mapping=True, verbose=False)["offset_mapping"]
chunks = []
for start in range(0, len(offsets), CHUNK_TOKENS - OVERLAP_TOKENS):
    end = min(start + CHUNK_TOKENS, len(offsets))
    char_start, char_end = offsets[start][0], offsets[end - 1][1]
    page = first_page + bisect.bisect_right(page_starts, char_start)  # 1-based PDF page
    chunks.append({"id": len(chunks), "page": page, "text": full_text[char_start:char_end]})
    if end == len(offsets):
        break

token_lengths = [len(tok(c["text"])["input_ids"]) for c in chunks]
# Check: no chunk may be cut off by the bi-encoder.
assert max(token_lengths) <= bi_encoder.max_seq_length, max(token_lengths)
print(f"{len(chunks)} chunks, longest = {max(token_lengths)} tokens (limit {bi_encoder.max_seq_length})")
print(f"\nExample: chunk 60 (page {chunks[60]['page']})\n" + textwrap.fill(chunks[60]["text"], 110))

337 chunks, longest = 203 tokens (limit 256)

Example: chunk 60 (page 32)
a wired connection to the PSTN that the verifier can call and dictate the out- of-band secret. For the
purposes of this definition, “wired connection” includes services such as cable providers that offer PSTN
services through other wired media and fiber via analog telephone adapters. For single-factor out-of-band
authenticators, if a secret is sent by the verifier to the out-of-band device, the device SHOULD NOT display
the authentication secret while it is locked by the owner. Rather, the device SHOULD require the presentation
and verification of a PIN, passcode, or biometric characteristic to view the secret. However, authenticators
SHOULD indicate the receipt of an authentication secret on a locked device. If the out-of-band authenticator
requests approval over the secondary communication channel rather than by presenting a secret that the
claimant transfers to the primary communication channel, it SHALL accep

## Step 3: Bi-encoder index (shared by both pipelines)

Each chunk is embedded once. Vectors are L2-normalised, so the inner product in `IndexFlatIP` **is** cosine similarity. A flat (exact) index is fine at this size; the tutorial notes exact search works up to roughly a million vectors.

In [4]:
chunk_vectors = bi_encoder.encode([c["text"] for c in chunks], normalize_embeddings=True,
                                  batch_size=32, show_progress_bar=False)
index = faiss.IndexFlatIP(chunk_vectors.shape[1])
index.add(chunk_vectors)

def retrieve(query, k=TOP_K):
    """Pipeline 1 search: returns [(chunk_id, cosine)] best first."""
    q = bi_encoder.encode([query], normalize_embeddings=True)
    scores, ids = index.search(q, k)
    return list(zip(ids[0].tolist(), scores[0].tolist()))

hits = retrieve("What is the minimum password length?")
assert len(hits) == TOP_K and all(a[1] >= b[1] for a, b in zip(hits, hits[1:]))
print(index.ntotal, "vectors indexed; sample hits:", [(i, round(s, 3)) for i, s in hits[:3]])

337 vectors indexed; sample hits: [(266, 0.693), (268, 0.621), (218, 0.601)]
